In [4]:
import os
import numpy as np
import tensorflow as tf
import sys
sys.path.insert(0, 'D:\KLASIFIKASI BERGANDA UNTUK OPTIMASI IMAGE DEBLURRING PADA CITRA DENGAN BLUR YANG BERVARIASI\library')
from utils import (
    preprocess_and_load_data,
    display_images,
    ssim,
    psnr,
    model_visualization,
    create_experiment_notes,
    tensorboard_callback,
    early_stopping_callback,
    checkpoint_callback,
    save_image
    )

## **Data Collecting**


In [5]:
blur_type = "gaussian_blur"
blur_level = "low"

In [6]:
x_dir = rf"D:\KLASIFIKASI BERGANDA UNTUK OPTIMASI IMAGE DEBLURRING PADA CITRA DENGAN BLUR YANG BERVARIASI\dataset\For Deblurring\blur_severity_level\{blur_type}\train\X\{blur_level}"
y_dir = rf"D:\KLASIFIKASI BERGANDA UNTUK OPTIMASI IMAGE DEBLURRING PADA CITRA DENGAN BLUR YANG BERVARIASI\dataset\For Deblurring\blur_severity_level\{blur_type}\train\Y\{blur_level}"

In [7]:
from sklearn.model_selection import train_test_split
# Get the list of filenames in the directories
x_filenames = os.listdir(x_dir)
y_filenames = os.listdir(y_dir)

# Split the dataset into train and test sets (adjust test_size and random_state as needed)
x_train_filenames, x_test_filenames, y_train_filenames, y_test_filenames = (
    train_test_split(x_filenames, y_filenames, test_size=0.2)
)

# Verify the lengths of the train and test sets
print("Number of training samples:", len(x_train_filenames))
print("Number of testing samples:", len(x_test_filenames))

Number of training samples: 8000
Number of testing samples: 2000


## **Data Preparation**


### **Load and preprocess training and testing data**


In [ ]:
with tf.device("/device:CPU:0"):
    x_train = preprocess_and_load_data(x_dir, x_train_filenames)
    y_train= preprocess_and_load_data(y_dir, y_train_filenames)

In [ ]:
with tf.device("/device:CPU:0"):
    x_test = preprocess_and_load_data(x_dir, x_test_filenames)
    y_test = preprocess_and_load_data(y_dir, y_test_filenames)

### **Display some examples of original Image and Blur Image**


In [ ]:
labels = ["Original Image", "Blurry Image"]
with tf.device("/device:CPU:0"):
    rng = np.random.default_rng()
    random_indices = rng.choice(len(x_train_filenames), size=10)
    display_images([y_train[random_indices], x_train[random_indices]], labels=labels,figsize=(25, 5))

## **Modelling**


### **Create the autoencoder model**


In [8]:
with tf.device("/device:CPU:0"):
    from model_architecture.UNet_Architecture import unet
    parent_dir = rf"D:\KLASIFIKASI BERGANDA UNTUK OPTIMASI IMAGE DEBLURRING PADA CITRA DENGAN BLUR YANG BERVARIASI\facial_image_deblurring\experiments\{blur_type}_experiments"
    model_name = f"{blur_type}_{blur_level}"
    learning_rate = 0.0001
    optimizer = tf.optimizers.Adam(learning_rate=learning_rate)
    model = unet(optimizer=optimizer, input_shape=(128, 128, 3))

In [3]:
model_visualization(model, parent_dir, model_name, figsize=(10,10))

NameError: name 'model_visualization' is not defined

### **Model Callbacks**

In [9]:
with tf.device("/device:CPU:0"):
    checkpoint, checkpoint_path = checkpoint_callback(model_name, parent_dir)
    early_stopping = early_stopping_callback(patience=25)
    log_dir, tensorboard = tensorboard_callback(parent_dir, model_name)

## **Model Training**


### **Train the model with blurry training data**


In [10]:
epochs = 100
batch_size = 8
validation_split = 0.2

In [ ]:
import time
start_time = time.time()
history = model.fit(
    x=x_train,
    y=y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=validation_split,
    callbacks=[early_stopping, checkpoint,tensorboard],
)
end_time = time.time()

execution_time = end_time - start_time

### **Plot the Loss, SSIM, and PSNR Graph**


In [ ]:
import matplotlib.pyplot as plt

# Plot Training Loss and Validation Loss
plt.figure(figsize=(18, 6))
plt.subplot(131)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training Loss and Validation Loss")

# Plot Training SSIM and Validation SSIM
plt.subplot(132)
plt.plot(history.history["ssim"], label="SSIM")
plt.plot(history.history["val_ssim"], label="Validation SSIM")
plt.xlabel("Epoch")
plt.ylabel("SSIM")
plt.legend()
plt.title("Training SSIM and Validation SSIM")

# Plot Training PSNR and Validation PSNR
plt.subplot(133)
plt.plot(history.history["psnr"], label="PSNR")
plt.plot(history.history["val_psnr"], label="Validation PSNR")
plt.xlabel("Epoch")
plt.ylabel("PSNR")
plt.legend()
plt.title("Training PSNR and Validation PSNR")

plt.show()

## **Model Evaluation**


In [13]:
# Load the saved model with custom metric functions
with tf.device("/device:CPU:0"):
    best_model = tf.keras.models.load_model(
        checkpoint_path,
        custom_objects={"ssim": ssim, "psnr": psnr},
    )

In [ ]:
with tf.device("/device:CPU:0"):
    # Evaluate the model on the test data
    evaluation = best_model.evaluate(x_test, y_test, batch_size=16)

    # Extract the metrics from the results
    loss, ssim_value, psnr_value = evaluation[0], evaluation[1], evaluation[2]

    # Display the results
    print(f"Test Loss (MSE): {loss}")
    print(f"Test SSIM: {ssim_value}")
    print(f"Test PSNR: {psnr_value}")

### **Experiment Logs**


In [15]:
with tf.device("/device:CPU:0"):
    model_summary_text = []
    for layer in model.layers:
        layer_summary = (
            f"{layer.name} ({layer.__class__.__name__}) {layer.output_shape}"
        )
        model_summary_text.append(layer_summary)

    model_summary_text = "<br/>".join(model_summary_text)

    experiment_notes = create_experiment_notes(
        model_name=model_name,
        model=model_summary_text,
        total_train_data=len(x_train_filenames),
        total_test_data=len(x_test_filenames),
        blur_type=blur_type,
        learning_rate=learning_rate,
        optimizer=optimizer.__class__.__name__,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        loss=loss,
        ssim_value=ssim_value,
        psnr_value=psnr_value,
    )

    text_file_writer = tf.summary.create_file_writer(log_dir + "/text/")
    with text_file_writer.as_default():
        tf.summary.text(log_dir, experiment_notes, step=0)

### **Display test, noisy test data, and predictions image**


In [ ]:
with tf.device("/device:CPU:0"):
    rng = np.random.default_rng()
    random_indices = rng.choice(len(x_test_filenames), size=10)
    labels = ["Original Image", "Blurry Image", "Reconstruction"]
    predictions = best_model(x_test[random_indices])
    display_images(
        [y_test[random_indices], x_test[random_indices], predictions],
        labels,
        figsize=(20, 7),
    )

In [17]:
from utils import shutdown

shutdown(1)